# Standard BKT — Model 1.1

Leave-one-participant-out evaluation of the standard BKT model. Each participant's predictions come from a model trained on the other 25 participants.

## Model specification

Model 1.1 fits one classic BKT chain for each of five knowledge components (KCs). Each chain learns initial mastery ($L_0$), learning transition ($T$), guess ($g$), and slip ($s$), with no forgetting. Question correctness is assigned to every designed KC for that question; the annotated cell-level labels are not used. For a question requiring multiple KCs, their predicted correctness probabilities are averaged.

## 1. Load data

In [1]:
from pathlib import Path
import pandas as pd

from scripts.data import load_data
from scripts.evaluator import Evaluator
from scripts.model_1_1 import Model_1_1

DATA_PATH = Path("data/data_annotated.csv")
SEED = 42
data = load_data(DATA_PATH)

## 2. Run evaluation

Evaluation uses 26-fold leave-one-participant-out cross-validation. In each fold, the model trains on 25 participants and predicts all 12 questions for the remaining participant. Pooling the held-out predictions produces 312 out-of-sample predictions. Each model uses five EM restarts and random seed 42.

In [2]:
evaluation = Evaluator(
    Model_1_1,
    data,
    model_kwargs={"n_restarts": 5},
    seed=SEED,
).run()

### 2.1 Out-of-fold metrics

The dataset contains 200 correct responses (64.1%) and 112 wrong responses (35.9%). An always-correct classifier would obtain 64.1% accuracy and an F1 score of 0.781, providing baselines for interpreting the model.

In [3]:
STANDARD_METRICS = ["auc", 'auprc_wrong', "accuracy", 'bal_acc', "f1", "log_loss", "n"]
metrics = pd.Series({name: evaluation.metrics[name] for name in STANDARD_METRICS}, name="value").to_frame()
metrics

,value
auc,0.597545
auprc_wrong,0.518238
accuracy,0.705128
bal_acc,0.597143
f1,0.809917
log_loss,0.628234
n,312.000000


**Interpretation.** Model 1.1 achieves AUC 0.598, accuracy 70.5%, F1 0.810, and log loss 0.628. Accuracy is 6.4 percentage points above the always-correct baseline. AUC indicates modest discrimination, while log loss improves slightly over the constant base-rate benchmark of 0.653. F1 treats `correct` as the positive class and is therefore helped by correct responses being the majority.

### 2.2 Confusion matrix

In [4]:
prediction_labels = evaluation.predictions.assign(
    predicted=lambda frame: (frame["p_pred"] >= 0.5).map({True: "correct", False: "wrong"}),
    actual=lambda frame: frame["y_true"].map({1: "correct", 0: "wrong"}),
)

confusion_matrix = pd.crosstab(
    prediction_labels["predicted"],
    prediction_labels["actual"],
    rownames=["predicted"],
    colnames=["actual"],
).reindex(index=["correct", "wrong"], columns=["correct", "wrong"], fill_value=0)
confusion_matrix

actual,correct,wrong
predicted,,
correct,196,88
wrong,4,24


**Interpretation.** The model identifies 196 of 200 correct responses (98.0% sensitivity) but only 24 of 112 wrong responses (21.4% specificity). The 88 false positives show that it strongly favors predicting `correct`.

### 2.3 Parameter ranges across folds

| Parameter | Meaning |
|---|---|
| $L_0$ | Probability that the KC is mastered before its first opportunity |
| $T$ | Probability of transitioning from unmastered to mastered after an opportunity |
| Guess | Probability of a correct response while unmastered |
| Slip | Probability of an incorrect response while mastered |

The table reports the minimum, maximum, and range across the 26 fitted fold models. These ranges are **not confidence intervals**: they describe how estimates change when a different participant is excluded from training.

In [5]:
fold_parameters = pd.DataFrame(
    [
        {
            "held_out_participant": participant_id,
            "kc": kc,
            "parameter": parameter,
            "value": value,
        }
        for participant_id, model in evaluation.fold_models.items()
        for kc, chain in model.chains.items()
        for parameter, value in {
            "L0": chain.L0,
            "T": chain.T,
            "guess": chain.g,
            "slip": chain.s,
        }.items()
    ]
)

parameter_ranges = (
    fold_parameters.groupby(["kc", "parameter"])["value"]
    .agg(mean="mean", minimum="min", maximum="max")
    .reset_index()
)
parameter_ranges["range"] = parameter_ranges["maximum"] - parameter_ranges["minimum"]
parameter_ranges

,kc,parameter,mean,minimum,maximum,range
0,kc1_sample_space,L0,0.757473,0.741848,0.800789,0.058942
1,kc1_sample_space,T,0.000100,0.000100,0.000100,0.000000
2,kc1_sample_space,guess,0.238067,0.207579,0.300000,0.092421
3,kc1_sample_space,slip,0.223329,0.209573,0.239590,0.030017
4,kc2_conditioning,L0,0.546697,0.486779,0.611231,0.124451
5,kc2_conditioning,T,0.000100,0.000100,0.000100,0.000000
6,kc2_conditioning,guess,0.300000,0.300000,0.300000,0.000000
7,kc2_conditioning,slip,0.246883,0.214961,0.288730,0.073769
8,kc3_joint_chain,L0,0.622330,0.577431,0.718969,0.141538
9,kc3_joint_chain,T,0.000100,0.000100,0.000100,0.000000


**Interpretation.** The transition parameter reaches its lower bound of 0.0001 for every KC in every fold. Under this model, the data provide essentially no evidence of within-session learning; predictions are driven primarily by initial mastery, guessing, and slipping. Guess reaches its upper bound of 0.30 for conditioning and approaches it for other KCs, so some estimates are constraint-sensitive. The table summarizes fold-specific models used for prediction; it is not a single model fitted to the complete dataset.

## 3. Limitations and reproducibility

The study contains 26 participants, and the reported results do not include uncertainty intervals. Pooled metrics treat all 312 responses equally even though responses from the same participant are related. Parameter identification is limited by the small number of opportunities per participant and KC. Results use `data/data_annotated.csv`, exclude Q0 warm-up rows through `load_data`, and use random seed 42.